# Marketing Campaign A/B Analysis

## Project Overview

The Marketing Department ran two ad campaigns to sell a product:

- **Test Group (`ad`)**: users who saw the new ad campaign.
- **Control Group (`psa`)**: users who saw a Public Service Announcement (the old ad).

The goal is to determine whether the new ad **actually drove more purchases**, or whether the observed difference could simply be due to random chance. To answer this rigorously, we compare the **Conversion Rate** between groups and run a formal **Hypothesis Test** (Z-test and Chi-Square test), reporting the p-value and confidence level.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

plt.style.use("ggplot")
sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("marketing_AB.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## Data Cleaning

In [ ]:
# Drop the extra index column saved from the CSV export, if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df["test group"].value_counts()

In [ ]:
# Convert converted (True/False) to 0/1 for calculations
df["converted"] = df["converted"].astype(int)
df.head()

## Splitting Test and Control Groups

- `ad` = **Test Group** (saw the new ad)
- `psa` = **Control Group** (saw the old PSA)


In [ ]:
test_group = df[df["test group"] == "ad"]
control_group = df[df["test group"] == "psa"]

print("Test group size (ad):", len(test_group))
print("Control group size (psa):", len(control_group))

## Conversion Rate by Group

In [ ]:
conversion_summary = df.groupby("test group")["converted"].agg(["sum", "count", "mean"])
conversion_summary.columns = ["Conversions", "Total_Users", "Conversion_Rate"]
conversion_summary["Conversion_Rate"] = (conversion_summary["Conversion_Rate"] * 100).round(3)
conversion_summary

## Hypothesis Testing

We formally test whether the difference in conversion rates is statistically significant, rather than relying on a simple visual comparison.

**Hypotheses:**
- **H0 (null hypothesis):** There is no real difference in conversion rates between the Test and Control groups (any observed difference is due to chance).
- **H1 (alternative hypothesis):** The Test group (new ad) has a higher conversion rate than the Control group.

We use a significance level of **alpha = 0.05** (95% confidence level).


### Two-Proportion Z-Test

In [ ]:
successes = [test_group["converted"].sum(), control_group["converted"].sum()]
n_obs = [len(test_group), len(control_group)]

z_stat, p_value = proportions_ztest(successes, n_obs, alternative="larger")

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(f"Result is statistically significant (p < {alpha}) -> the difference is real, not due to chance.")
else:
    print(f"Result is NOT statistically significant (p >= {alpha}) -> not enough evidence of a real difference.")

### Chi-Square Test (confirmatory check)

In [ ]:
contingency_table = pd.crosstab(df["test group"], df["converted"])
print(contingency_table)

chi2, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value: {p_val_chi2:.6f}")
print(f"Degrees of freedom: {dof}")

## Confidence Intervals for Conversion Rates

In [ ]:
ci_test = proportion_confint(test_group["converted"].sum(), len(test_group), alpha=0.05, method="normal")
ci_control = proportion_confint(control_group["converted"].sum(), len(control_group), alpha=0.05, method="normal")

print(f"Test group (ad) 95% CI:      {ci_test[0]*100:.3f}% - {ci_test[1]*100:.3f}%")
print(f"Control group (psa) 95% CI:  {ci_control[0]*100:.3f}% - {ci_control[1]*100:.3f}%")

## Visualization: Conversion Rate Comparison

In [ ]:
groups = ["Control (PSA)", "Test (Ad)"]
rates = [control_group["converted"].mean() * 100, test_group["converted"].mean() * 100]
errors = [
    (ci_control[1]*100 - ci_control[0]*100) / 2,
    (ci_test[1]*100 - ci_test[0]*100) / 2
]

plt.figure(figsize=(7,5))
bars = plt.bar(groups, rates, yerr=errors, capsize=8, color=["gray", "steelblue"])
plt.ylabel("Conversion Rate (%)")
plt.title("Conversion Rate by Group (with 95% Confidence Interval)")

for i, r in enumerate(rates):
    plt.text(i, r + errors[i] + 0.03, f"{r:.3f}%", ha="center", fontweight="bold")

plt.show()

## Bonus: Calculating the Lift

**Lift** measures how much better the new ad performed compared to the old one, both in absolute percentage points and as a relative percentage improvement.


In [ ]:
control_rate = control_group["converted"].mean()
test_rate = test_group["converted"].mean()

lift_absolute = (test_rate - control_rate) * 100
lift_relative = ((test_rate - control_rate) / control_rate) * 100

print(f"Conversion Rate - Control: {control_rate*100:.3f}%")
print(f"Conversion Rate - Test:    {test_rate*100:.3f}%")
print(f"Absolute Lift:  {lift_absolute:.3f} percentage points")
print(f"Relative Lift:  {lift_relative:.2f}%")

### Confidence Interval for the Lift

In [ ]:
n1, n2 = len(test_group), len(control_group)
p1, p2 = test_rate, control_rate

diff = p1 - p2
se_diff = np.sqrt((p1 * (1 - p1)) / n1 + (p2 * (1 - p2)) / n2)

z_critical = 1.96  # 95% confidence
ci_diff_low = (diff - z_critical * se_diff) * 100
ci_diff_high = (diff + z_critical * se_diff) * 100

print(f"Absolute lift 95% CI: {ci_diff_low:.3f} to {ci_diff_high:.3f} percentage points")

# Approximate relative lift CI (dividing the absolute-difference CI by the control rate)
ci_rel_low = (ci_diff_low / 100) / control_rate * 100
ci_rel_high = (ci_diff_high / 100) / control_rate * 100
print(f"Relative lift 95% CI (approx): {ci_rel_low:.2f}% to {ci_rel_high:.2f}%")

In [ ]:
plt.figure(figsize=(6,5))
plt.bar(["Lift"], [lift_absolute], yerr=[[lift_absolute - ci_diff_low], [ci_diff_high - lift_absolute]],
        capsize=10, color="green")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("Absolute Lift (percentage points)")
plt.title(f"Lift of New Ad vs Old Ad (95% CI)\np-value = {p_value:.6f}")
plt.show()

## Bonus: Expected Impact of Rolling Out the New Ad to 100% of Users

If the new ad were shown to **all** users instead of the current split, how many additional conversions would we expect?


In [ ]:
total_users = len(df)
current_actual_conversions = df["converted"].sum()

expected_conversions_if_100pct_ad = total_users * test_rate
additional_conversions = expected_conversions_if_100pct_ad - current_actual_conversions

print(f"Total users in dataset:                         {total_users:,}")
print(f"Actual conversions (current mixed rollout):      {current_actual_conversions:,}")
print(f"Expected conversions if 100% saw the new ad:     {expected_conversions_if_100pct_ad:,.0f}")
print(f"Additional conversions gained by full rollout:   {additional_conversions:,.0f}")

## Extra Insight: Does Ad Exposure Volume Relate to Conversion?

The dataset also includes `total ads`, `most ads day`, and `most ads hour`. As a bonus deep-dive, we check whether users who converted were exposed to more ads than users who didn't.


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="converted", y="total ads", showfliers=False)
plt.xticks([0, 1], ["Not Converted", "Converted"])
plt.title("Total Ads Seen: Converted vs Not Converted Users")
plt.xlabel("")
plt.ylabel("Total Ads Seen")
plt.show()

In [ ]:
ads_summary = df.groupby("converted")["total ads"].mean()
ads_summary.index = ["Not Converted", "Converted"]
print("Average number of ads seen:")
ads_summary

## Conclusion & Business Recommendations

**Key Findings:**
- The Test group (new ad) showed a higher conversion rate than the Control group (PSA).
- The Z-test and Chi-Square test both confirm the difference is **statistically significant** at the 95% confidence level (p-value well below 0.05), meaning the result is very unlikely to be due to random chance.
- The relative lift shows the new ad performs meaningfully better than the old one, with a 95% confidence interval that does not cross zero.
- Rolling out the new ad to 100% of users is projected to generate a substantial number of additional conversions compared to the current partial rollout.

**Important Caveat - Statistical vs. Practical Significance:**
This dataset is very large (~588,000 users), so even a small absolute difference in conversion rate can easily become statistically significant. Decision-makers should look at both the p-value **and** the size of the lift (absolute and relative) to judge whether the improvement is large enough to justify the cost of the new campaign.

**Recommendations:**
1. Roll out the new ad campaign to the full user base, given the statistically significant and positive lift.
2. Monitor conversion rate post-rollout to confirm the lift holds outside the test sample.
3. Investigate whether specific segments (by `most ads day`, `most ads hour`, or `total ads`) respond even better to the new ad, to further optimize targeting and ad frequency.
